## 07 · Literature ingestion (WP1 / D1.2)

Thin driver over `gif.zenodo` + `gif.literature`. Downloads the two June-2026 BioFairNet uploads and turns them into tidy, validated CSVs:

| Source | DOI | License |
|---|---|---|
| Literature Results – Full List | [10.5281/zenodo.20743706](https://doi.org/10.5281/zenodo.20743706) | CC-BY-4.0 |
| Codebook (manually coded) | [10.5281/zenodo.20744025](https://doi.org/10.5281/zenodo.20744025) | CC-BY-4.0 |

**Attribution:** Guerreschi, Lomuscio & Albanese (BioFairNet WP1). Keep `isDerivedFrom` links to both DOIs in any derived release.

Outputs: `data/processed/literature/{papers,codes_long,papers_coded}.csv`. The join between the two files is title-based (the codebook's own `paper_id` drifts after blank spreadsheet rows) and is validated — mismatches are reported, never silently misassigned.

In [ ]:
import sys
from pathlib import Path

# Make the repo importable: `helper` at the root, `gif` under `src/`.
repo_root = Path.cwd().parents[0]
for p in (str(repo_root), str(repo_root / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

from gif.zenodo import download_record
from gif.literature import (
    prepare_literature, D12_FULL_LIST_DOI, D12_CODEBOOK_DOI,
)

### Download from Zenodo

Files are MD5-verified against the record checksums; existing verified files are skipped, so re-running is cheap.

In [ ]:
dest = repo_root / "data" / "external"
full_files  = download_record(D12_FULL_LIST_DOI, dest)
coded_files = download_record(D12_CODEBOOK_DOI, dest)
print("Full list:", full_files[0].name)
print("Codebook :", coded_files[0].name)

### Prepare tidy CSVs

In [ ]:
report = prepare_literature(
    full_files[0], coded_files[0],
    repo_root / "data" / "processed" / "literature",
)
j = report["join"]
print(f"Papers: {j['papers']} | coded rows aligned: {j['coded']} "
      f"| title mismatches: {j['title_mismatches']}")
print(f"Codes assigned: {report['codes_assigned']} across {report['code_dimensions']}")

### Preview

In [ ]:
import pandas as pd

papers = pd.read_csv(repo_root / "data/processed/literature/papers.csv")
codes  = pd.read_csv(repo_root / "data/processed/literature/codes_long.csv")
print("Sector breakdown:")
print(papers["sector_tag"].value_counts().to_string())
print("\nTop barriers (title+keyword coding):")
print(codes[(codes.dimension == "barriers") & (codes.source == "title_kw")]
      ["code"].value_counts().head(10).to_string())
papers.head(3)